# 04 — Prediction Intervals & Risk Scoring

**Pipeline stage 4 of 5**

## Objective
Two things a bare point forecast can't tell a farmer: *how wide is the
uncertainty*, and *how much should I trust this specific crop×market's
number at all*. This notebook produces both:

1. **Prediction intervals per tier**, widening from the 7–14 day tier to the
   60–90 day tier, using conformal prediction (distribution-free, doesn't
   assume Gaussian residuals — appropriate for thin, noisy agricultural
   markets).
2. **A risk/volatility score per crop×market pair**, combining backtest
   error (notebook 03) with raw price volatility (notebook 01), so the
   dashboard/app can visually flag low-confidence commodities instead of
   presenting every forecast with equal authority.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = ROOT / "data" / "processed"

backtest_results = pd.read_parquet(PROCESSED_DIR / "backtest_results.parquet")
volatility = pd.read_parquet(PROCESSED_DIR / "commodity_volatility.parquet")
backtest_results.head()

## 1. Conformal-style prediction intervals per tier

Split-conformal approach: use held-out fold residuals from the walk-forward backtest (notebook 03) directly as the empirical error distribution, rather than assuming normality. The interval half-width is the residual distribution's upper quantile — this is what `quant/intervals.py` should implement in production, backed by whichever conformal library (e.g. MAPIE) is chosen at that point.

In [ ]:
CONFIDENCE_LEVEL = 0.80  # 80% coverage — wide enough to be honest, narrow enough to be useful

def empirical_interval_halfwidth(mae_mean: float, mae_std: float, confidence: float = CONFIDENCE_LEVEL) -> float:
    """
    Approximates a conformal interval half-width from fold-level MAE mean/std
    when raw residuals aren't retained. In production, replace this with the
    true empirical residual quantile from stored fold predictions.
    """
    from scipy.stats import norm
    z = norm.ppf(0.5 + confidence / 2)
    return mae_mean + z * (mae_std if not np.isnan(mae_std) else 0)

backtest_results["interval_halfwidth_ugx"] = backtest_results.apply(
    lambda r: empirical_interval_halfwidth(r["mae_mean"], r["mae_std"]), axis=1
)
backtest_results.groupby("tier_label")["interval_halfwidth_ugx"].mean().reindex(["7-14 days", "30 days", "60-90 days"])

## 2. Risk score per crop×market

Combines: (a) backtest MAPE — how wrong the model has been historically, and (b) raw commodity price volatility (CV) from notebook 01 — how noisy the underlying series is, independent of the model. A crop×market can be low-risk on one dimension and high on the other; scoring both avoids hiding that.

In [ ]:
risk = (
    backtest_results
    .groupby(["crop_enc", "market_enc"])
    .agg(mape_mean=("mape_mean", "mean"), n_folds=("n_folds", "sum"))
    .reset_index()
)

# crop_enc needs to map back to commodity name to join volatility; encoders saved in notebook 02
import joblib
encoders = joblib.load(PROCESSED_DIR / "feature_encoders.pkl")
le_crop = encoders["le_crop"]
risk["commodity"] = le_crop.inverse_transform(risk["crop_enc"])

risk = risk.merge(volatility[["cv"]].reset_index(), on="commodity", how="left")

# Normalize both components to 0-1, average into a single risk score (higher = riskier)
risk["mape_norm"] = (risk["mape_mean"] - risk["mape_mean"].min()) / (risk["mape_mean"].max() - risk["mape_mean"].min())
risk["cv_norm"] = (risk["cv"] - risk["cv"].min()) / (risk["cv"].max() - risk["cv"].min())
risk["risk_score"] = (risk["mape_norm"].fillna(0) + risk["cv_norm"].fillna(0)) / 2

risk_report = risk.sort_values("risk_score", ascending=False)
risk_report.to_parquet(PROCESSED_DIR / "risk_scores.parquet", index=False)
risk_report[["commodity", "market_enc", "mape_mean", "cv", "risk_score", "n_folds"]].head(15)

## 3. Confidence label mapping

What the app/dashboard actually displays — a 3-band label rather than a raw score, since farmers need an at-a-glance signal, not a number to interpret.

In [ ]:
def confidence_label(score: float) -> str:
    if score < 0.33:
        return "High confidence"
    elif score < 0.66:
        return "Moderate confidence"
    return "Low confidence — treat as directional only"

risk_report["confidence_label"] = risk_report["risk_score"].apply(confidence_label)
risk_report["confidence_label"].value_counts()

## Output

- `data/processed/risk_scores.parquet` — per crop×market risk score and confidence label
- Tier-level interval half-widths, folded into `backtest_results` (already saved in notebook 03, extended here)

**Next:** `05_model_export.ipynb`